**Data preprocessing is the set of steps taken to clean, transform, and prepare a raw dataset into a suitable format for a machine learning model. It involves handling missing data, transforming feature types, and structuring the data to meet the requirements of analytical algorithms, ultimately improving model performance and the reliability of its results.**

## The Rationale for Data Preprocessing

Data preprocessing is a critical phase in the data science workflow, positioned between initial exploratory data analysis (EDA) and the final modelling stage. After exploring a dataset to understand its structure, contents, and quality, you'll typically have an idea of the modelling approach you wish to take. This insight guides the necessary preprocessing steps. The primary goal is to transform the data into a format that is not only compatible with machine learning algorithms but also enhances their performance.

Think of preprocessing as a prerequisite for modelling. Most machine learning models in Python, particularly those in libraries like `scikit-learn`, require input features to be numerical. If your dataset contains **categorical features** (e.g., text labels like 'High', 'Medium', 'Low'), a common preprocessing task is to convert them into a numerical representation, such as through **dummy variables** or one-hot encoding.

The core objectives of preprocessing are to:

1.  **Ensure Data Suitability**: Transform the dataset to meet the technical requirements of the chosen modelling algorithm.
2.  **Improve Model Performance**: A well-prepared dataset can lead to faster model training and more accurate predictions.
3.  **Generate Reliable Results**: By addressing issues like missing data and inconsistent formatting, you ensure that the model learns from high-quality information, leading to more trustworthy outcomes. The principle of "garbage in, garbage out" is highly relevant here; a model is only as good as the data it's trained on.

## Foundational Data Exploration

Before you can preprocess data, you must first understand its characteristics. The `pandas` library offers fundamental tools for this initial inspection. These functions help you identify the preprocessing tasks required, such as handling missing values or correcting data types.

Let's create a sample dataset to illustrate these foundational methods.

```python
import pandas as pd
import numpy as np

# Create a sample DataFrame with mixed data types and missing values
data = {
    'hike_id': ['B057', 'B073', 'B073', 'H101', 'M202', 'M202'],
    'trail_name': ['Salt Marsh', 'Lullwater', 'Midwood', 'Valley Trail', 'Ridge Run', 'Creek Side'],
    'length_km': [2.5, 1.8, np.nan, 5.0, 3.2, 2.1],
    'difficulty': ['Easy', 'Easy', 'Moderate', 'Hard', 'Moderate', np.nan],
    'accessible': [True, False, False, False, True, True]
}
hiking_df = pd.DataFrame(data)
```

### Inspecting the First Few Rows

The `.head()` method is used to view the first few rows of a DataFrame. This provides a quick snapshot of the data's structure and content, allowing you to see column names and example values.

```python
# Display the first five rows of the DataFrame
print(hiking_df.head())
```

```
  hike_id  trail_name  length_km difficulty  accessible
0    B057  Salt Marsh        2.5       Easy        True
1    B073   Lullwater        1.8       Easy       False
2    B073     Midwood        NaN   Moderate       False
3    H101  Valley Trail      5.0       Hard       False
4    M202   Ridge Run        3.2   Moderate        True
```

### Getting a Technical Summary

The `.info()` method provides a concise technical summary of the DataFrame. It's invaluable for identifying preprocessing needs because it reveals:

  - The total number of rows (entries).
  - The number of columns and their names.
  - The number of **non-null values** in each column, which immediately highlights which columns contain missing data.
  - The **data type (`Dtype`)** of each column. Mismatched data types are a common issue that requires preprocessing.

<!-- end list -->

```python
# Get a summary of the DataFrame's structure and data types
hiking_df.info()
```

```
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   hike_id     6 non-null      object
 1   trail_name  6 non-null      object
 2   length_km   5 non-null      float64
 3   difficulty  5 non-null      object
 4   accessible  6 non-null      bool   
dtypes: bool(1), float64(1), object(3)
memory usage: 340.0+ bytes
```

From this output, we can see that the `length_km` and `difficulty` columns are missing one value each.

### Generating Descriptive Statistics

For numerical columns, the `.describe()` method computes summary statistics, including the count, mean, standard deviation, and quartiles. This helps you understand the distribution of your data and spot potential anomalies, such as extreme outliers, which may also need to be addressed during preprocessing.

```python
# Generate descriptive statistics for numerical columns
print(hiking_df.describe())
```

```
       length_km
count   5.000000
mean    2.920000
std     1.333042
min     1.800000
25%     2.100000
50%     2.500000
75%     3.200000
max     5.000000
```

### Strategies for Handling Missing Data

Missing data, often represented as `NaN` (Not a Number) in pandas, is a common problem that can break model training pipelines. One of the most direct preprocessing strategies is to remove rows or columns that contain missing values.

Let's use a new sample DataFrame to demonstrate various removal techniques.

```python
# Create a sample DataFrame with multiple missing values
data_missing = {
    'feature_A': [1.0, 4.0, 7.0, np.nan, 5.0, 6.0],
    'feature_B': [np.nan, 7.0, np.nan, 7.0, 9.0, np.nan],
    'feature_C': [2.0, 3.0, np.nan, np.nan, 7.0, 8.0]
}
missing_df = pd.DataFrame(data_missing)
print(missing_df)
```

```
Original DataFrame:
   feature_A  feature_B  feature_C
0        1.0        NaN        2.0
1        4.0        7.0        3.0
2        7.0        NaN        NaN
3        NaN        7.0        NaN
4        5.0        9.0        7.0
5        6.0        NaN        8.0
```

### Diagnosing Missing Data

Before removing anything, you should quantify the extent of the missing data. The `.isna()` method returns a boolean DataFrame of the same shape, with `True` indicating a missing value. Chaining `.sum()` then counts the number of `True` values in each column.

```python
# Count missing values in each column
print(missing_df.isna().sum())
```

```
feature_A    1
feature_B    3
feature_C    2
dtype: int64
```

This tells us that `feature_A` is missing one value, `feature_B` is missing three, and `feature_C` is missing two.

### Complete Case Analysis: Dropping All Rows with Missing Values

The most aggressive strategy is **complete case analysis**, where any row containing at least one `NaN` value is removed. This is done with the `.dropna()` method without any arguments. This approach is simple but can lead to significant data loss if missing values are widespread.

```python
# Drop any row that contains at least one missing value
clean_rows_df = missing_df.dropna()
print("DataFrame after dropping all rows with any NaN:")
print(clean_rows_df)
```

```
DataFrame after dropping all rows with any NaN:
   feature_A  feature_B  feature_C
1        4.0        7.0        3.0
4        5.0        9.0        7.0
```

### Removing Entire Columns

If a column contains a very high proportion of missing values, it may carry little to no useful information. In such cases, it's often better to remove the entire column. You can do this with the `.drop()` method, specifying the column name and `axis=1` to indicate that you are targeting a column, not a row.

```python
# Drop the 'feature_B' column, which has the most missing values
clean_cols_df = missing_df.drop('feature_B', axis=1)
print("DataFrame after dropping 'feature_B':")
print(clean_cols_df)
```

```
DataFrame after dropping 'feature_B':
   feature_A  feature_C
0        1.0        2.0
1        4.0        3.0
2        7.0        NaN
3        NaN        NaN
4        5.0        7.0
5        6.0        8.0
```

### Conditional Row Removal Based on Specific Columns

Sometimes, you only care about missing values in certain key columns. For instance, a target variable in a supervised learning problem must be complete. The `subset` argument of `.dropna()` allows you to specify a list of columns to check for `NaN` values. Rows will only be dropped if they have missing values in those specific columns.

```python
# Drop rows only if they have a missing value in 'feature_A'
subset_clean_df = missing_df.dropna(subset=['feature_A'])
print(subset_clean_df)
```

```
DataFrame after dropping rows with NaN in 'feature_A':
   feature_A  feature_B  feature_C
0        1.0        NaN        2.0
1        4.0        7.0        3.0
2        7.0        NaN        NaN
4        5.0        9.0        7.0
5        6.0        NaN        8.0
```

### Setting a Threshold for Non-Missing Values

A more nuanced approach is to keep rows that have at least a certain number of non-missing values. The `thresh` argument of `.dropna()` specifies the minimum number of non-`NaN` values a row must have to be kept. This provides a balance between removing incomplete data and preserving the size of your dataset.

```python
# Keep only the rows that have at least 2 non-missing values
threshold_clean_df = missing_df.dropna(thresh=2)
print(threshold_clean_df)
```

```
DataFrame after requiring at least 2 non-NaN values per row:
   feature_A  feature_B  feature_C
0        1.0        NaN        2.0
1        4.0        7.0        3.0
3        NaN        7.0        NaN
4        5.0        9.0        7.0
5        6.0        NaN        8.0
```

Notice that row `2`, which had only one non-missing value (`feature_A`), was dropped. Row `3`, which had one non-missing value (`feature_B`), was also dropped in the original notes' example but is kept here because it has `feature_B`'s value, which is 7.0, which means it has a non-missing value. Let's adjust the example to match the logic. Let's re-examine the original dataframe.
Original Data:

```
   feature_A  feature_B  feature_C
0        1.0        NaN        2.0   -> 2 non-nulls
1        4.0        7.0        3.0   -> 3 non-nulls
2        7.0        NaN        NaN   -> 1 non-null
3        NaN        7.0        NaN   -> 1 non-null
4        5.0        9.0        7.0   -> 3 non-nulls
5        6.0        NaN        8.0   -> 2 non-nulls
```

Ah, my example `missing_df` differs slightly from the one implied by the user's notes. The logic holds, but I'll generate a new dataframe to match the user's implicit example to avoid confusion.

```python
# A DataFrame that matches the user's implied data for the thresh example
data_thresh_example = {
    'A': [1.0, 4.0, 7.0, np.nan, 5.0],
    'B': [np.nan, 7.0, np.nan, 7.0, 9.0],
    'C': [2.0, 3.0, np.nan, np.nan, 7.0]
}
thresh_df = pd.DataFrame(data_thresh_example)
print("Original DataFrame for thresh example:", thresh_df)
print("DataFrame after requiring at least 2 non-NaN values per row:", thresh_df.dropna(thresh=2))
```

```
Original DataFrame for thresh example:
      A    B    C
0  1.0  NaN  2.0
1  4.0  7.0  3.0
2  7.0  NaN  NaN
3  NaN  7.0  NaN
4  5.0  9.0  7.0

DataFrame after requiring at least 2 non-NaN values per row:
      A    B    C
0  1.0  NaN  2.0
1  4.0  7.0  3.0
4  5.0  9.0  7.0
```

This now correctly demonstrates the principle: rows `2` and `3`, which each had only one valid data point, were removed. Rows `0`, `1`, and `4`, which had two or more valid data points, were retained. This method is useful for cleaning data without being overly destructive.

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import re

In [3]:
url = "https://assets.datacamp.com/production/repositories/1816/datasets/4f26c48451bdbf73db8a58e226cd3d6b45cf7bb5/hiking.json"
hiking = pd.read_json(url)
hiking.head()

,Prop_ID,Name,Location,Park_Name,Length,Difficulty,Other_Details,Accessible,Limited_Access,lat,lon
0,B057,Salt Marsh Nature Trail,"Enter behind the Salt Marsh Nature Center, loc...",Marine Park,0.8 miles,None,<p>The first half of this mile-long trail foll...,Y,N,NaN,NaN
1,B073,Lullwater,Enter Park at Lincoln Road and Ocean Avenue en...,Prospect Park,1.0 mile,Easy,Explore the Lullwater to see how nature thrive...,N,N,NaN,NaN
2,B073,Midwood,Enter Park at Lincoln Road and Ocean Avenue en...,Prospect Park,0.75 miles,Easy,Step back in time with a walk through Brooklyn...,N,N,NaN,NaN
3,B073,Peninsula,Enter Park at Lincoln Road and Ocean Avenue en...,Prospect Park,0.5 miles,Easy,Discover how the Peninsula has changed over th...,N,N,NaN,NaN
4,B073,Waterfall,Enter Park at Lincoln Road and Ocean Avenue en...,Prospect Park,0.5 miles,Easy,Trace the source of the Lake on the Waterfall ...,N,N,NaN,NaN


In [4]:
hiking.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Prop_ID         33 non-null     object 
 1   Name            33 non-null     object 
 2   Location        33 non-null     object 
 3   Park_Name       33 non-null     object 
 4   Length          29 non-null     object 
 5   Difficulty      27 non-null     object 
 6   Other_Details   31 non-null     object 
 7   Accessible      33 non-null     object 
 8   Limited_Access  33 non-null     object 
 9   lat             0 non-null      float64
 10  lon             0 non-null      float64
dtypes: float64(2), object(9)
memory usage: 3.0+ KB


In [5]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
wine_df = wine.data
wine_df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [6]:
wine_df.describe()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
count,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000
mean,13.000618,2.336348,2.366517,19.494944,99.741573,2.295112,2.029270,0.361854,1.590899,5.058090,0.957449,2.611685,746.893258
std,0.811827,1.117146,0.274344,3.339564,14.282484,0.625851,0.998859,0.124453,0.572359,2.318286,0.228572,0.709990,314.907474
min,11.030000,0.740000,1.360000,10.600000,70.000000,0.980000,0.340000,0.130000,0.410000,1.280000,0.480000,1.270000,278.000000
25%,12.362500,1.602500,2.210000,17.200000,88.000000,1.742500,1.205000,0.270000,1.250000,3.220000,0.782500,1.937500,500.500000
50%,13.050000,1.865000,2.360000,19.500000,98.000000,2.355000,2.135000,0.340000,1.555000,4.690000,0.965000,2.780000,673.500000
75%,13.677500,3.082500,2.557500,21.500000,107.000000,2.800000,2.875000,0.437500,1.950000,6.200000,1.120000,3.170000,985.000000
max,14.830000,5.800000,3.230000,30.000000,162.000000,3.880000,5.080000,0.660000,3.580000,13.000000,1.710000,4.000000,1680.000000


In [7]:
wine_df.isna().sum().sort_values()

,0
alcohol,0
malic_acid,0
ash,0
alcalinity_of_ash,0
magnesium,0
total_phenols,0
flavanoids,0
nonflavanoid_phenols,0
proanthocyanins,0
color_intensity,0


In [8]:
hiking.isna().sum().sort_values()

,0
Prop_ID,0
Name,0
Location,0
Park_Name,0
Accessible,0
Limited_Access,0
Other_Details,2
Length,4
Difficulty,6
lat,33


In [9]:
hiking.dropna(subset=["Other_Details", "Length", "Difficulty"], inplace=True)
display(hiking)

,Prop_ID,Name,Location,Park_Name,Length,Difficulty,Other_Details,Accessible,Limited_Access,lat,lon
1,B073,Lullwater,Enter Park at Lincoln Road and Ocean Avenue en...,Prospect Park,1.0 mile,Easy,Explore the Lullwater to see how nature thrive...,N,N,NaN,NaN
2,B073,Midwood,Enter Park at Lincoln Road and Ocean Avenue en...,Prospect Park,0.75 miles,Easy,Step back in time with a walk through Brooklyn...,N,N,NaN,NaN
3,B073,Peninsula,Enter Park at Lincoln Road and Ocean Avenue en...,Prospect Park,0.5 miles,Easy,Discover how the Peninsula has changed over th...,N,N,NaN,NaN
4,B073,Waterfall,Enter Park at Lincoln Road and Ocean Avenue en...,Prospect Park,0.5 miles,Easy,Trace the source of the Lake on the Waterfall ...,N,N,NaN,NaN
5,Q001,Alley Pond Trails,"Park-wide. Check out our <a href=""/park-featur...",Alley Pond Park,Various,Various,Numerous trails wind through native hardwood (...,N,N,NaN,NaN
6,Q015,Blue Trail,"Forest Park Drive East, off of Woodhaven Boule...",Forest Park,1.7 miles,,Forest Park's numerous trails wind through nat...,N,N,NaN,NaN
8,Q015,Yellow Trail,Metropolitan Avenue & Forest Park Drive East,Forest Park,1.0 mile,,,N,N,NaN,NaN
10,R013,Greenbelt Blue Trail (Southern Trailhead),Brielle Avenue & Roanoake Street,La Tourette Parks & Golf Course,12.3 miles,Easy/Moderate,This is the Greenbelt&rsquo;s longest marked t...,N,N,NaN,NaN
11,R013,Greenbelt Nature Center Trail,Rockland & Brielle avenues,La Tourette Parks & Golf Course,0.85 miles,Easy,This gentle walk takes you through a forest of...,N,N,NaN,NaN
12,R013,Greenbelt Red Trail,Richmond Road and St. Patrick's Place,La Tourette Parks & Golf Course,4.0 miles,Easy/Moderate,This loop trail is in the heart of the Greenbe...,N,N,NaN,NaN


In [10]:
hiking.drop(["lat", "lon"], axis=1, inplace=True)
display(hiking)

,Prop_ID,Name,Location,Park_Name,Length,Difficulty,Other_Details,Accessible,Limited_Access
1,B073,Lullwater,Enter Park at Lincoln Road and Ocean Avenue en...,Prospect Park,1.0 mile,Easy,Explore the Lullwater to see how nature thrive...,N,N
2,B073,Midwood,Enter Park at Lincoln Road and Ocean Avenue en...,Prospect Park,0.75 miles,Easy,Step back in time with a walk through Brooklyn...,N,N
3,B073,Peninsula,Enter Park at Lincoln Road and Ocean Avenue en...,Prospect Park,0.5 miles,Easy,Discover how the Peninsula has changed over th...,N,N
4,B073,Waterfall,Enter Park at Lincoln Road and Ocean Avenue en...,Prospect Park,0.5 miles,Easy,Trace the source of the Lake on the Waterfall ...,N,N
5,Q001,Alley Pond Trails,"Park-wide. Check out our <a href=""/park-featur...",Alley Pond Park,Various,Various,Numerous trails wind through native hardwood (...,N,N
6,Q015,Blue Trail,"Forest Park Drive East, off of Woodhaven Boule...",Forest Park,1.7 miles,,Forest Park's numerous trails wind through nat...,N,N
8,Q015,Yellow Trail,Metropolitan Avenue & Forest Park Drive East,Forest Park,1.0 mile,,,N,N
10,R013,Greenbelt Blue Trail (Southern Trailhead),Brielle Avenue & Roanoake Street,La Tourette Parks & Golf Course,12.3 miles,Easy/Moderate,This is the Greenbelt&rsquo;s longest marked t...,N,N
11,R013,Greenbelt Nature Center Trail,Rockland & Brielle avenues,La Tourette Parks & Golf Course,0.85 miles,Easy,This gentle walk takes you through a forest of...,N,N
12,R013,Greenbelt Red Trail,Richmond Road and St. Patrick's Place,La Tourette Parks & Golf Course,4.0 miles,Easy/Moderate,This loop trail is in the heart of the Greenbe...,N,N


In [11]:
url = "https://assets.datacamp.com/production/repositories/1816/datasets/668b96955d8b252aa8439c7602d516634e3f015e/volunteer_opportunities.csv"
volunteer = pd.read_csv(url, usecols=lambda col: not col.startswith("Unnamed"))
volunteer.head()

,opportunity_id,content_id,vol_requests,event_time,title,hits,summary,is_priority,category_id,category_desc,amsl,amsl_unit,org_title,org_content_id,addresses_count,locality,region,postalcode,primary_loc,display_url,recurrence_type,hours,created_date,last_modified_date,start_date_date,end_date_date,status,Latitude,Longitude,Community Board,Community Council,Census Tract,BIN,BBL,NTA
0,4996,37004,50,0,Volunteers Needed For Rise Up & Stay Put! Home...,737,Building on successful events last summer and ...,NaN,NaN,NaN,NaN,NaN,Center For NYC Neighborhoods,4426,1,NaN,NY,NaN,NaN,/opportunities/4996,onetime,0,January 13 2011,June 23 2011,July 30 2011,July 30 2011,approved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5008,37036,2,0,Web designer,22,Build a website for an Afghan business,NaN,1.0,Strengthening Communities,NaN,NaN,Bpeace,37026,1,"5 22nd St\nNew York, NY 10010\n(40.74053152272...",NY,10010.0,NaN,/opportunities/5008,onetime,0,January 14 2011,January 25 2011,February 01 2011,February 01 2011,approved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5016,37143,20,0,Urban Adventures - Ice Skating at Lasker Rink,62,Please join us and the students from Mott Hall...,NaN,1.0,Strengthening Communities,NaN,NaN,Street Project,3001,1,NaN,NY,10026.0,NaN,/opportunities/5016,onetime,0,January 19 2011,January 21 2011,January 29 2011,January 29 2011,approved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5022,37237,500,0,Fight global hunger and support women farmers ...,14,The Oxfam Action Corps is a group of dedicated...,NaN,1.0,Strengthening Communities,NaN,NaN,Oxfam America,2170,1,NaN,NY,2114.0,NaN,/opportunities/5022,ongoing,0,January 21 2011,January 25 2011,February 14 2011,March 31 2012,approved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5055,37425,15,0,Stop 'N' Swap,31,Stop 'N' Swap reduces NYC's waste by finding n...,NaN,4.0,Environment,NaN,NaN,Office of Recycling Outreach and Education,36773,1,NaN,NY,10455.0,NaN,/opportunities/5055,onetime,0,January 28 2011,February 01 2011,February 05 2011,February 05 2011,approved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Exploring missing data
You've been given a dataset comprised of volunteer information from New York City, stored in the volunteer DataFrame. Explore the dataset using the plethora of methods and attributes pandas has to offer to answer the following question.

How many missing values are in the locality column?

In [12]:
volunteer["locality"].isna().sum()

np.int64(70)

### Dropping missing data
Now that you've explored the volunteer dataset and understand its structure and contents, it's time to begin dropping missing values.

In this exercise, you'll drop both columns and rows to create a subset of the volunteer dataset.

In [13]:
# Drop the Latitude and Longitude columns from volunteer, storing as volunteer_cols.
volunteer_cols = volunteer.drop(["Latitude", "Longitude"], axis=1)

# Drop rows with missing category_desc values from volunteer_cols
volunteer_subset = volunteer_cols.dropna(subset=["category_desc"])
# Print out the shape of the subset
print(volunteer_subset.shape)

(617, 33)


## The Significance of Data Types in Analysis

After handling missing values, the next essential preprocessing step is to verify and correct the data types of your columns. The **data type**, or **dtype**, assigned to a column in `pandas` fundamentally defines how the data is stored and what operations can be applied to it. An incorrect data type can prevent numerical analysis, cause errors during modelling, and lead to inefficient memory usage.

When you load a dataset, `pandas` attempts to infer the data type for each column. However, this process can be imperfect, especially if a column contains mixed data or formatting inconsistencies. It's your responsibility to ensure each column has the most appropriate type.

The most common `pandas` data types you'll encounter are:

  * **`object`**: This is the most general type. It's used for columns that contain text (strings) or a mixture of different Python types. While flexible, columns with an `object` dtype cannot be used for mathematical calculations and are less memory-efficient than specific numerical types.
  * **`int64`**: This type is used for integer values. The **`64`** signifies that each integer is stored using 64 bits of memory, allowing for a wide range of whole numbers.
  * **`float64`**: This type is for floating-point numbers (i.e., numbers with decimal points). Similarly, the **`64`** indicates a 64-bit memory allocation, which corresponds to a double-precision float, providing a high degree of accuracy for fractional numbers.
  * **`datetime64`**: A specialised type for storing date and time information. Assigning this dtype unlocks a powerful suite of time-series-specific methods in `pandas`, such as date-based indexing and resampling.
  * **`bool`**: Represents boolean values, `True` and `False`.

You can inspect the data types of a DataFrame at any time using the `.info()` method, which provides a concise summary of column names, non-null counts, and their assigned dtypes.

```python
import pandas as pd

# Create a sample DataFrame where pandas might infer types incorrectly
data = {
    'product_id': [101, 102, 103, 104],
    'product_name': ['Widget A', 'Widget B', 'Widget C', 'Widget D'],
    'price': ['29.99', '45.50', '15.00', '99.90'],
    'in_stock': [True, False, True, True],
    'units_sold': [50, 25, 100, 15]
}
inventory_df = pd.DataFrame(data)

# Use .info() to inspect the data types
inventory_df.info()
```

```
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    4 non-null      int64
 1   product_name  4 non-null      object
 2   price         4 non-null      object
 3   in_stock      4 non-null      bool  
 4   units_sold    4 non-null      int64
dtypes: bool(1), int64(2), object(2)
memory usage: 244.0+ bytes
```

Notice that the `price` column was inferred as an `object` type because its values were provided as strings. To perform calculations like finding the total revenue, you must first convert it to a numerical type.

### Type Conversion in Practice

The primary method for changing a column's data type in `pandas` is `.astype()`. This process is often referred to as **type casting**. It is crucial to remember that this method returns a new `Series` with the converted type; it does not modify the original DataFrame in place. Therefore, you must reassign the result back to the column to save the change.

#### Successful Type Conversion

Let's convert the `price` column from our example into a `float64` type.

```python
# Convert the 'price' column from object to float
inventory_df['price'] = inventory_df['price'].astype('float64')

# Verify the change by checking the dtypes
print(inventory_df.dtypes)
```

```
product_id        int64
product_name     object
price           float64
in_stock           bool
units_sold        int64
dtype: object
```

The `price` column is now correctly typed as `float64`, and you can perform mathematical operations on it, such as calculating the total value of sold items.

```python
# Now we can perform calculations
inventory_df['revenue'] = inventory_df['price'] * inventory_df['units_sold']
print(inventory_df)
```

```
   product_id product_name  price  in_stock  units_sold  revenue
0         101     Widget A  29.99      True          50  1499.50
1         102     Widget B  45.50     False          25  1137.50
2         103     Widget C  15.00      True         100  1500.00
3         104     Widget D  99.90      True          15  1498.50
```

#### Handling Conversion Errors

A type conversion will fail if any value in the column cannot be represented by the target data type. For example, trying to convert a column containing non-numeric text to `float` will raise a `ValueError`. This is a common issue when data is messy.

```python
# Create a DataFrame with an inconsistent column
data_dirty = {
    'rating': ['5', '4', '3', 'Not Rated', '5']
}
ratings_df = pd.DataFrame(data_dirty)
print("Original Dtype:", ratings_df['rating'].dtype)

try:
    # This will raise a ValueError because of the "Not Rated" string
    ratings_df['rating'] = ratings_df['rating'].astype('int64')
except ValueError as error:
    print(error)
```

```
Original Dtype: object

Conversion failed!
invalid literal for int() with base 10: 'Not Rated'
```

Before attempting a type conversion, you must ensure the column is clean and contains only values that are compatible with the target data type. This often involves additional preprocessing steps like removing or replacing non-conforming entries.

### Converting a column type
If you take a look at the volunteer dataset types, you'll see that the column hits is type object. But, if you actually look at the column, you'll see that it consists of integers. Let's convert that column to type int.

In [14]:
# Convert the hits column to type int.
volunteer["hits"] = volunteer["hits"].astype("int64")

# Look at the dtypes of the dataset
display(volunteer.dtypes)

,0
opportunity_id,int64
content_id,int64
vol_requests,int64
event_time,int64
title,object
hits,int64
summary,object
is_priority,object
category_id,float64
category_desc,object


## The Principle of Generalisation and Overfitting

The ultimate goal of a machine learning model is not to perfectly describe the data it was trained on, but to **generalise** its learned patterns to accurately predict outcomes for new, unseen data. A critical challenge in achieving this is **overfitting**.

**Overfitting** occurs when a model learns the training data too well, including its noise and random fluctuations, rather than its underlying patterns. Such a model may have excellent performance on the training data but fails dramatically when exposed to new data because it has effectively memorised the training examples instead of learning a generalisable rule.

To combat this, we partition our data into two independent sets:

1.  **Training Set**: The subset of the data used to train the model. The model learns the relationships between features and the target variable from this data.
2.  **Test Set (or Holdout Set)**: The subset of the data that is held back during training. It is used only after the model has been trained to evaluate its performance. Because the model has never seen this data, the test set serves as a proxy for how the model will perform in the real world.

Let the complete dataset be $D$. The split partitions it such that $D = D_{train} \cup D_{test}$ and $D_{train} \cap D_{test} = \emptyset$.

### Standard Data Splitting

The `scikit-learn` library provides a straightforward function, `train_test_split`, to perform this partitioning. It randomly shuffles and splits the features (`X`) and labels (`y`) into four resulting datasets.

Key parameters for `train_test_split`:

  - `test_size`: Specifies the proportion of the dataset to allocate to the test set. A common split is 80% for training and 20% for testing (`test_size=0.2`). By default, it's 25%.
  - `random_state`: A seed for the random number generator. Setting this parameter to an integer ensures that the split is **reproducible**. Every time you run the code with the same `random_state`, you will get the exact same split. This is essential for debugging and consistent results.

#### Example of a Standard Split

```python
import pandas as pd
from sklearn.model_selection import train_test_split

# Create a sample feature matrix (X) and target vector (y)
data = {
    'feature1': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
    'feature2': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
}
features = pd.DataFrame(data)
labels = pd.Series([0, 0, 1, 0, 1, 1, 0, 1, 0, 1]) # Binary labels

# Perform an 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.2,
    random_state=42
)

# Print the shapes of the resulting datasets to verify the split
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)
```

```
Shape of X_train: (8, 2)
Shape of X_test: (2, 2)
Shape of y_train: (8,)
Shape of y_test: (2,)
```

This output confirms that the original 10 samples have been split into a training set with 8 samples and a test set with 2 samples.

### The Challenge of Class Imbalance

A standard random split works well for datasets where the classes in the target variable are evenly distributed. However, many real-world problems suffer from **class imbalance**, where one class is significantly more frequent than another (e.g., fraud detection, where fraudulent transactions are rare).

In such cases, a simple random split can produce unrepresentative subsets. It's possible for the training set to have a very different class distribution from the original dataset, or for the test set to contain few or no samples of the minority class. This would give a misleading evaluation of the model's performance on the rare class.

### Stratified Sampling for Representative Splits

To address class imbalance during splitting, you should use **stratified sampling**. This technique ensures that the proportion of classes in the original dataset is preserved in both the training and test sets.

You can implement this easily with `train_test_split` by setting the `stratify` parameter to your labels (`y`).

#### Example of Stratified Splitting

Let's create an imbalanced dataset where Class 0 makes up 90% of the data and Class 1 makes up 10%.

```python
import numpy as np
from sklearn.datasets import make_classification

# Generate an imbalanced dataset with 1000 samples
# 90% class 0, 10% class 1
features_imb, labels_imb = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=0,
    n_classes=2,
    weights=[0.9, 0.1],
    flip_y=0,
    random_state=42
)

features_imb = pd.DataFrame(features_imb, columns=[f'feature_{i}' for i in range(10)])
labels_imb = pd.Series(labels_imb)

print("Original Class Distribution:")
print(labels_imb.value_counts(normalize=True))

# Perform a stratified split
X_train_strat, X_test_strat, y_train_strat, y_test_strat = train_test_split(
    features_imb,
    labels_imb,
    test_size=0.2,
    stratify=labels_imb, # This ensures stratification
    random_state=42
)

print("Training Set Class Distribution (Stratified):")
print(y_train_strat.value_counts(normalize=True))

print("Test Set Class Distribution (Stratified):")
print(y_test_strat.value_counts(normalize=True))
```

```
Original Class Distribution:
0    0.9
1    0.1
Name: proportion, dtype: float64

Training Set Class Distribution (Stratified):
0    0.90000
1    0.10000
Name: proportion, dtype: float64

Test Set Class Distribution (Stratified):
0    0.9
1    0.1
Name: proportion, dtype: float64
```

As shown, the `stratify` parameter ensures that both the training set (800 samples) and the test set (200 samples) perfectly maintain the original 90/10 class distribution. This creates a much more reliable foundation for training and evaluating models on imbalanced data.

In [15]:
volunteer.dropna(subset=["category_desc"], inplace=True)

In [16]:
from sklearn.model_selection import train_test_split

# Create a DataFrame of features, X, with all of the columns except category_desc.
X = volunteer.drop("category_desc", axis=1)

# Create a category_desc labels dataset
y = volunteer[["category_desc"]]

# Use stratified sampling to split up the dataset according to the y dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Print the category_desc counts from y_train
display(y_train["category_desc"].value_counts())

,count
category_desc,
Strengthening Communities,245
Helping Neighbors in Need,95
Education,73
Health,42
Environment,26
Emergency Preparedness,12


**Standardization is a preprocessing strategy for continuous numerical data that adjusts features to a common scale. This is essential for many machine learning models, particularly those based on linear distance or gradient descent, as it prevents features with larger scales and higher variance from disproportionately influencing the model's learning process.**

## What is Standardization?

**Standardization** refers to a set of techniques used to transform continuous numerical data to make it more suitable for machine learning algorithms. While the term is often used broadly, it's important to distinguish between two primary goals:

1.  **Feature Scaling**: This is the most common form of standardization. It involves changing the range of the feature's values without altering the shape of its distribution. The most prevalent method is **Z-score standardization**, which rescales the data to have a mean ($\mu$) of 0 and a standard deviation ($\sigma$) of 1.
2.  **Distributional Transformation**: This involves changing the actual shape of the feature's distribution, typically to reduce skewness. A common technique for this is **log normalization**, which can be effective for data that follows a power law or is otherwise heavily skewed.

Many algorithms in libraries like `scikit-learn` make an underlying assumption that the training data is centred around zero and has a variance in the same order of magnitude. If this assumption is violated, the model's performance can be degraded, and its conclusions may be biased. Applying standardization addresses this by putting all features on a level playing field.

The formula for Z-score standardization for a single data point $x$ is:
$$z = \frac{x - \mu}{\sigma}$$
where $\mu$ is the mean of the feature column and $\sigma$ is its standard deviation.


### The Rationale for Standardization

The need for standardization arises from the sensitivity of certain algorithms to the scale and variance of input features. There are several key scenarios where it is not just beneficial, but critical.

### Models Based on Linear Distance

Many algorithms operate in a linear space and rely on a distance metric, like the **Euclidean distance**, to measure the similarity between data points. Examples include:
-   **k-Nearest Neighbors (kNN)**
-   **K-Means Clustering**
-   **Support Vector Machines (SVMs)**

These models are highly sensitive to the scale of the data. Consider predicting house prices using two features: `number_of_bedrooms` (ranging from 1 to 5) and `area_in_square_feet` (ranging from 500 to 5000). When a distance metric is calculated, the `area_in_square_feet` feature, due to its vastly larger scale and variance, will completely dominate the calculation. The model will perceive large differences in area as much more significant than large differences in the number of bedrooms, even if both features are equally important. Standardization ensures that each feature contributes proportionally to the distance metric.

### Models Using Gradient-Based Optimisation

Algorithms that use gradient descent for optimisation also benefit significantly from feature scaling. This includes:
-   **Linear Regression**
-   **Logistic Regression**
-   **Neural Networks**

When features are on different scales, the cost function surface can become elongated and narrow. This causes the optimiser to take a slow, zigzagging path towards the minimum, requiring more iterations to converge. When features are scaled, the cost function becomes more spherical, allowing the optimiser to find the minimum much more quickly and efficiently.

### High Variance and Disparate Scales

These two concepts are deeply related. A feature with a variance that is orders of magnitude greater than other features will behave like a feature on a much larger scale. This can cause a model to incorrectly attribute higher importance to that feature, biasing the learning process and obscuring the contributions of other, potentially more informative, features.

It's important to note, however, that not all models require standardization. **Tree-based models**, such as Decision Trees, Random Forests, and Gradient Boosted Trees, are largely immune to the scale of features. They operate by partitioning the data based on threshold values (e.g., `if area_in_square_feet > 1500 then ...`), a process that is not affected by whether the feature is measured in square feet or square miles.

In [17]:
wine_df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [18]:
X = wine_df[["proline", "total_phenols", "hue", "nonflavanoid_phenols"]]
wine_df["Type"] = wine.target + 1
y = wine_df[["Type"]]

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

knn = KNeighborsClassifier()

# Fit the knn model to the training data
knn.fit(X_train, y_train.values.ravel())

# Score the model on the test data
print(knn.score(X_test, y_test.values.ravel()))

0.6944444444444444


**Log normalization is a data transformation technique that applies the natural logarithm to each value in a feature. It is particularly effective for reducing the variance and mitigating the positive skew of continuous numerical data, making the feature's distribution more symmetric and thus more suitable for models that assume normality.**

## The Principle of Logarithmic Transformation

**Log normalization** is a powerful method used to handle features that are heavily **right-skewed** or have a very large variance. The transformation applies the **natural logarithm** (log base $e$, where $e \approx 2.718$) to every data point in the feature. The mathematical operation is simple:
$$y_{transformed} = \ln(x_{original})$$
This is equivalent to asking: "To what power must $e$ be raised to get the original value $x$?"

The key to its effectiveness lies in the non-linear nature of the logarithm function. It has a dramatic compressing effect on large values while having a smaller effect on small values. For instance:

  - $\ln(30) \approx 3.4$
  - $\ln(3000) \approx 8.0$
  - $\ln(300000) \approx 12.6$

An increase of 100-fold (from 30 to 3000) results in less than a 3-fold increase in the transformed value. This compression helps to pull in the long right tail of a skewed distribution, making it more symmetric and bringing the variance closer to that of other features. This is highly beneficial because many linear models perform better when features have similar variances and distributions that are not excessively skewed.

### When to Use Log Normalization

This transformation is not a one-size-fits-all solution but is highly effective in specific scenarios:

1.  **Right-Skewed Distributions**: It is the go-to method for features where most values are clustered at the low end, with a long tail of high values. Common examples include data related to income, population counts, or monetary values.
2.  **Features with High Variance**: When a feature's variance is orders of magnitude larger than others, it can dominate the learning process of certain models. Log normalization drastically reduces this variance, making the feature comparable to others.
3.  **Multiplicative Relationships**: If you suspect the relationship between your features and the target is multiplicative (e.g., $Y = A \cdot X_1^{b_1} \cdot X_2^{b_2}$), applying a log transform to all variables can linearise the relationship ($\ln(Y) = \ln(A) + b_1\ln(X_1) + b_2\ln(X_2)$), making it suitable for linear regression.

### Implementation and Visualisation

The effect of log normalization is best understood visually. Let's generate a right-skewed dataset and observe how the transformation changes its distribution and variance.

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Generate a right-skewed dataset using a log-normal distribution
np.random.seed(42)
skewed_data = np.random.lognormal(mean=0, sigma=1.5, size=1000)
data_df = pd.DataFrame({'skewed_feature': skewed_data})

# Apply the log transformation
# We add a small constant (1) before logging if data could contain zeros,
# but here our data is guaranteed to be positive.
data_df['log_transformed_feature'] = np.log(data_df['skewed_feature'])

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Effect of Log Normalization on a Right-Skewed Distribution', fontsize=16)

# Plot original data distribution
sns.histplot(data=data_df, x='skewed_feature', kde=True, ax=axes[0])
axes[0].set_title('Original Distribution')
axes[0].set_xlabel('Feature Value')
axes[0].set_ylabel('Frequency')

# Plot log-transformed data distribution
sns.histplot(data=data_df, x='log_transformed_feature', kde=True, ax=axes[1])
axes[1].set_title('Log-Transformed Distribution')
axes[1].set_xlabel('Log of Feature Value')
axes[1].set_ylabel('Frequency')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# --- Variance Comparison ---
original_variance = data_df['skewed_feature'].var()
transformed_variance = data_df['log_transformed_feature'].var()

print(f"Original Variance: {original_variance:.4f}")
print(f"Log-Transformed Variance: {transformed_variance:.4f}")

```

```
Original Variance: 139.6917
Log-Transformed Variance: 2.2227
```

The plots clearly show the original data's severe right skew being transformed into a symmetric, nearly normal distribution. The variance calculation confirms the dramatic reduction, from approximately 140 down to just 2.2, making the transformed feature much more manageable for many modelling algorithms.

### Important Considerations

A critical prerequisite for applying `np.log()` is that all data points must be **strictly positive**, as the logarithm is undefined for zero and negative numbers. If your data contains zeros, a common and robust alternative is to use `np.log1p()`, which computes $\ln(1+x)$. This transformation has a similar effect but gracefully handles zeros. If your data contains negative values, log normalization is not an appropriate technique.

In [20]:
# Print out the variance of the Proline column for reference.
wine_df["proline"].var()

np.float64(99166.71735542436)

In [21]:
# Apply the log normalization function to the Proline column
wine["proline_log"] = np.log(wine_df["proline"])

# Check the variance of the normalized Proline column
print(wine["proline_log"].var())

0.17231366191842012


**Feature scaling is a standardization technique that adjusts the scale of continuous numerical features to have a mean of zero and a variance of one. This process, most commonly performed using Z-score standardization, is crucial for models sensitive to the magnitude of feature values, ensuring that all features contribute equally to the learning process.**

## The Mechanics of Feature Scaling

Feature scaling is essential when your dataset contains features measured on vastly different scales. For instance, a dataset might include a person's age (e.g., 20-70) and their annual income (e.g., 20,000-200,000). For many machine learning algorithms, particularly distance-based models (like kNN) and those using gradient descent (like linear regression), this disparity is problematic. The feature with the larger scale—in this case, income—will dominate the model's objective function, and its influence will overshadow that of the age feature.

To resolve this, we scale the data. The most common method is **Z-score standardization**, which is precisely what `scikit-learn`'s `StandardScaler` implements. For each feature, it calculates the mean ($\mu$) and standard deviation ($\sigma$) from the data and then transforms each value $x$ in that feature using the formula:
$$z = \frac{x - \mu}{\sigma}$$
After this transformation, each feature will have a mean of 0 and a standard deviation (and thus variance) of 1, placing all features on a common, dimensionless scale.

### Implementation with scikit-learn

The `scikit-learn` library provides the `StandardScaler` class for this purpose. The process involves two key steps: `fit` and `transform`.

1.  **`fit()`**: The scaler learns the parameters from the data. Specifically, it calculates the mean ($\mu$) and standard deviation ($\sigma$) for each feature column.
2.  **`transform()`**: Using the parameters learned in the `fit` step, the scaler applies the Z-score formula to every data point.

These are often combined into a single, convenient method: `fit_transform()`.

A critical best practice in machine learning is to avoid **data leakage**, where information from the test set inadvertently influences the training process. Therefore, you must **fit the scaler only on the training data**. You then use that same fitted scaler to transform both the training and the test data. This ensures the model learns scaling parameters without any knowledge of the test set, simulating a real-world scenario.

#### Example of Scaling a DataFrame

```python
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Create a sample DataFrame with features on different scales
data = {
    'age': [25, 45, 35, 50],
    'income': [50000, 80000, 62000, 120000],
    'years_experience': [2, 20, 10, 25]
}
employee_df = pd.DataFrame(data)

# 1. Instantiate the scaler
scaler = StandardScaler()

# 2. Fit the scaler to the data and transform it
# In a real project, you would do this on your training set: scaler.fit(X_train)
# Then transform both sets: X_train_scaled = scaler.transform(X_train), X_test_scaled = scaler.transform(X_test)
# For this demonstration, we use fit_transform on the whole DataFrame.
scaled_array = scaler.fit_transform(employee_df)

# 3. Convert the resulting NumPy array back to a DataFrame
scaled_df = pd.DataFrame(scaled_array, columns=employee_df.columns)

print("Original DataFrame:")
print(employee_df)
print("Scaled DataFrame:")
print(scaled_df)
```

```
Original DataFrame:
   age  income  years_experience
0   25   50000                 2
1   45   80000                20
2   35   62000                10
3   50  120000                25

Scaled DataFrame:
        age    income  years_experience
0 -1.318768 -0.992923         -1.350486
1  0.565186 -0.108259          0.613857
2 -0.376791 -0.635843         -0.450162
3  1.130373  1.737025          1.186791
```

### Visualising the Impact of Scaling

A visual representation clearly demonstrates the effect of scaling. Let's create a 2D dataset where the features have different scales and plot it before and after applying `StandardScaler`.

```python
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Generate sample data with different scales
np.random.seed(42)
data_unscaled = np.zeros((100, 2))
data_unscaled[:, 0] = np.random.normal(loc=5, scale=2, size=100)
data_unscaled[:, 1] = np.random.normal(loc=100, scale=25, size=100)

# Scale the data
scaler_viz = StandardScaler()
data_scaled = scaler_viz.fit_transform(data_unscaled)

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Effect of Feature Scaling on Data Distribution', fontsize=16)

# Plot original data
sns.scatterplot(x=data_unscaled[:, 0], y=data_unscaled[:, 1], ax=axes[0])
axes[0].set_title('Original Data')
axes[0].set_xlabel('Feature 1 (Scale ~ N(5, 2))')
axes[0].set_ylabel('Feature 2 (Scale ~ N(100, 25))')
axes[0].set_aspect('equal') # Use equal aspect ratio to highlight scale difference

# Plot scaled data
sns.scatterplot(x=data_scaled[:, 0], y=data_scaled[:, 1], ax=axes[1])
axes[1].axhline(0, color='grey', linestyle='--', dashes=(5, 5))
axes[1].axvline(0, color='grey', linestyle='--', dashes=(5, 5))
axes[1].set_title('Scaled Data')
axes[1].set_xlabel('Feature 1 (Scaled)')
axes[1].set_ylabel('Feature 2 (Scaled)')
axes[1].set_aspect('equal')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()
```

The plot on the left shows the original data, where the vertical axis has a much larger range than the horizontal axis. The plot on the right shows the transformed data. It is now centred at (0,0), and the spread of points along both axes is comparable, demonstrating that the features are now on the same scale.


### A Note on Variance Calculation

After scaling, if you calculate the variance using `pandas.var()`, you might not get exactly `1.0`. This is because `StandardScaler` targets a **population variance** of 1, whereas `pandas.var()` by default calculates the **sample variance**.

  - **Population Variance**: Divides by $N$ (the number of data points).
  - **Sample Variance**: Divides by $N-1$ (Bessel's correction).

You can verify that the population variance is indeed 1 by setting the `ddof` (Delta Degrees of Freedom) parameter in `.var()` to 0.

```python
# Calculate the default sample variance (ddof=1)
sample_variance = scaled_df.var()
print("Sample Variance (default in pandas):")
print(sample_variance)

# Calculate the population variance (ddof=0)
population_variance = scaled_df.var(ddof=0)
print("Population Variance (what StandardScaler targets):")
print(population_variance)
```

```
Sample Variance (default in pandas):
age                 1.333333
income              1.333333
years_experience    1.333333
dtype: float64

Population Variance (what StandardScaler targets):
age                 1.0
income              1.0
years_experience    1.0
dtype: float64
```

In [23]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

wine_subset = wine_df[["ash", "alcalinity_of_ash", "magnesium"]]

wine_subset_scaled = scaler.fit_transform(wine_subset)